In [1]:
from sentence_transformers import SentenceTransformer
import torch
# Load embedding model
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("google/embeddinggemma-300m", device=device)

def create_embedding(texts):
    if type(texts) == list: # the input param is a list contains text chunks
        embeddings = model.encode(texts)
        return embeddings
    elif type(texts) == str: # just one text chunk
        embedding = model.encode([texts])
        return embedding
    else:
        raise ValueError("The texts paramters should be str or list[str]")

print(device)

C:\Users\admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuda


In [2]:
import os
dir_path = os.getcwd()
print("The directory of this script is:", dir_path)
root_path = os.path.dirname(dir_path)
print("The root directory is:", root_path)

The directory of this script is: d:\Documents\GitHub\NodeRAG\testing
The root directory is: d:\Documents\GitHub\NodeRAG


In [3]:
#LLM api call
from google import genai
with open(f"{root_path}/API_KEY.txt", "r", encoding="utf-8") as f:
    API_KEY = f.read()

def call_gemini(text):
    client = genai.Client(api_key = API_KEY)
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=text
    )
    return response.text, response.usage_metadata.total_token_count

In [4]:
import sys
import numpy as np
sys.path.append(root_path)
from testing.metrics.faithfulness import compute_faithfulness #Detect hallucinations: Measures how much the LLM answer is based on the retrieved context
"""
question: str,
answer: str,
contexts: List[str],
call_gemini,
max_retries: int = 5
"""
from testing.metrics.context_recall import compute_context_recall #Evaluate retrieval coverage: Measures how much of the reference (ground-truth) answer is supported by the retrieved context.
"""
question: str,
contexts: List[str],
reference_answer: str,
call_gemini,
max_retries: int = 5
"""
from testing.metrics.coverage import compute_coverage #Measures how completely a model’s response covers the factual content of a reference (ground-truth) answer.
"""
question: str,
reference: str,
response: str,
call_gemini,
max_retries: int = 5
"""
from testing.metrics.rouge import compute_rouge
#Calculate F-1 of lexical overlap by LCS: longest common subsequence (in terms of word order, adjacency not required)
#Precision = len(LCS) / len(Answer)
#Recall = len(LCS) / len(Ref)
"""
answer: str,
ground_truth: str,
rouge_type: str = "rougeL",
mode: str = "fmeasure"
"""
from testing.metrics.accuracy import compute_answer_accuracy
"""
question: str,
answer: str,
ground_truth: str,
call_gemini,
create_embedding,
weights: List[float] = [0.75, 0.25],
beta: float = 1.0,
max_retries: int = 5
"""
print("Import evaluation functions")


Import evaluation functions


In [5]:
import pandas as pd
medical_questions_answered_loaded = pd.read_parquet("data/medical_questions_answered.parquet")
relevant_questions = medical_questions_answered_loaded[medical_questions_answered_loaded["question_type"] == "Fact Retrieval"]
question_ids = relevant_questions['id'].to_list()
relevant_questions

,id,source,question,answer,question_type,evidence,evidence_relations,LLM_answer,LLM_context,LLM_tokens
0,Medical-73586ddc,Medical,What is the most common type of skin cancer?,Basal cell carcinoma (BCC) is the most common ...,Fact Retrieval,Basal cell carcinoma (BCC) is the most common ...,Basal cell carcinoma (BCC) is the most common ...,"Basal cell skin cancer, also known as basal ce...",Squamous cell skin cancer (CSCC) is the second...,24381.0
1,Medical-a8bad1cf,Medical,From which cell type does basal cell carcinoma...,Basal cell carcinoma arises from basal cells i...,Fact Retrieval,Basal cell carcinoma arises from basal cells.;...,BCC arises from basal cells in the lower part ...,Basal cell carcinoma arises when basal cells i...,"Basal cell skin cancer, also known as basal ce...",4113.0
2,Medical-422500d5,Medical,Which anatomical locations are most commonly a...,BCC most commonly develops in sun-exposed area...,Fact Retrieval,BCC most commonly develops in sun-exposed area...,BCC most commonly develops in sun-exposed area...,Basal cell carcinoma most commonly affects sun...,Basal cell skin cancer can affect anyone but i...,3514.0
3,Medical-6d2a190d,Medical,What is the primary risk factor for basal cell...,UV radiation exposure is a primary risk factor...,Fact Retrieval,UV radiation exposure is a primary risk factor...,UV radiation exposure is a primary risk factor...,The primary risk factor for basal cell carcino...,Basal cell skin cancer can affect anyone but i...,6428.0
4,Medical-5ad931db,Medical,How does fair skin affect the risk of developi...,Fair skin increases the risk of BCC.,Fact Retrieval,Fair skin increases the risk of BCC.,"Fair skin, light hair, and light eye color inc...",Fair skin increases the risk of developing Bas...,Basal cell skin cancer can affect anyone but i...,8562.0
...,...,...,...,...,...,...,...,...,...,...
1093,Medical-329de573,Medical,What should be discussed before starting breas...,Fertility preservation should be discussed bef...,Fact Retrieval,Fertility preservation should be discussed bef...,Fertility preservation should be discussed bef...,"Before starting breast cancer treatment, indiv...",Patients who wish to have children after cance...,27329.0
1094,Medical-e79a7e61,Medical,What patient factor is considered in treatment...,Performance status is considered in treatment ...,Fact Retrieval,Performance status is considered in treatment ...,Performance status is considered in treatment ...,Patient factors considered in treatment planni...,Treatment planning for patients starts with te...,28422.0
1095,Medical-883feb97,Medical,What does follow-up for breast cancer include?,"Follow-up includes imaging, physical exams, an...",Fact Retrieval,Follow-up for breast cancer includes imaging.;...,"Follow-up includes imaging, physical exams, an...","For breast cancer, follow-up and surveillance ...","""FOLLOW-UP AND SURVEILLANCE"" stands as the vig...",28392.0
1096,Medical-53f6f364,Medical,Which breast cancer subtype commonly presents ...,Inflammatory breast cancer (IBC) commonly pres...,Fact Retrieval,Inflammatory breast cancer (IBC) commonly pres...,Inflammatory breast cancer (IBC): Symptom: Red...,Inflammatory breast cancer (IBC) commonly pres...,Inflammatory breast cancer (IBC) is challengin...,6546.0


In [6]:
try:
    medical_questions_evaluated = pd.read_parquet("data/evaluations/fact_retrieval_evaluated.parquet")
    medical_questions_evaluated = (
        medical_questions_evaluated
        .set_index("qid")
        .to_dict(orient="index")
    )
except:
    medical_questions_evaluated = {}


medical_questions_evaluated

{}

In [7]:
def both_valid(a, b):
    return (
        a is not None and b is not None
        and not np.isnan(a)
        and not np.isnan(b)
    )

In [8]:
#accuracy
#rouge-L

from tqdm import tqdm
import time
sep = "\n\n" + "-"*100 + "\n\n"
MAX_RETRIES = 10
SLEEP_SEC = 30

def get_answers():
    for qid in tqdm(question_ids):
        if qid in medical_questions_evaluated:
            continue

        row = relevant_questions.loc[relevant_questions["id"] == qid].iloc[0]
        row_eval = {}

        question = row["question"]
        reference = row["answer"]
        answer = row["LLM_answer"]
        
        rouge_score = compute_rouge(answer, reference)
        accuracy, tokens = None, None
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                accuracy, tokens = compute_answer_accuracy(question, answer, reference, call_gemini, create_embedding)
                break
            except Exception as e:
                code = e.error.code if hasattr(e, "error") else e
                print(f"[ERROR] qid={qid} failed after {attempt} retries: {code}")
                if attempt == MAX_RETRIES:
                    return
                time.sleep(SLEEP_SEC * attempt)  #backoff
        if both_valid(accuracy, tokens):
            row_eval["accuracy"] = accuracy
            row_eval["rouge"] = rouge_score
            row_eval["tokens"] = tokens
            medical_questions_evaluated[qid] = row_eval
get_answers()


100%|██████████| 1098/1098 [5:47:22<00:00, 18.98s/it] 


In [9]:
medical_questions_evaluated

{'Medical-73586ddc': {'accuracy': 0.9964528679097717,
  'rouge': 0.7741935483870968,
  'tokens': 1799},
 'Medical-a8bad1cf': {'accuracy': 0.7241071759987783,
  'rouge': 0.6206896551724138,
  'tokens': 6419},
 'Medical-422500d5': {'accuracy': 0.7276801466441833,
  'rouge': 0.48000000000000004,
  'tokens': 7103},
 'Medical-6d2a190d': {'accuracy': 0.6030602305658799,
  'rouge': 0.19047619047619047,
  'tokens': 2423},
 'Medical-5ad931db': {'accuracy': 0.44477821671216744,
  'rouge': 0.2641509433962264,
  'tokens': 2763},
 'Medical-6b0f8bc4': {'accuracy': 0.725485593013854,
  'rouge': 0.2857142857142857,
  'tokens': 1526},
 'Medical-75c9949b': {'accuracy': 0.9880138634510376,
  'rouge': 0.56,
  'tokens': 1177},
 'Medical-ec091b24': {'accuracy': 0.5310109585163605,
  'rouge': 0.34782608695652173,
  'tokens': 2964},
 'Medical-8d39da2e': {'accuracy': 0.8448711335190278,
  'rouge': 0.41304347826086957,
  'tokens': 5387},
 'Medical-88c0b2ba': {'accuracy': 0.22807148098945618,
  'rouge': 0.153846

In [10]:
df_eval = (
    pd.DataFrame.from_dict(medical_questions_evaluated, orient="index")
      .reset_index()
      .rename(columns={"index": "qid"})
)

In [11]:
df_eval.to_parquet("data/evaluations/fact_retrieval_evaluated.parquet")
df_eval_loaded = pd.read_parquet("data/evaluations/fact_retrieval_evaluated.parquet")
df_eval_loaded

,qid,accuracy,rouge,tokens
0,Medical-73586ddc,0.996453,0.774194,1799
1,Medical-a8bad1cf,0.724107,0.620690,6419
2,Medical-422500d5,0.727680,0.480000,7103
3,Medical-6d2a190d,0.603060,0.190476,2423
4,Medical-5ad931db,0.444778,0.264151,2763
...,...,...,...,...
1093,Medical-329de573,0.468845,0.100000,4088
1094,Medical-e79a7e61,0.427511,0.200000,5180
1095,Medical-883feb97,0.929114,0.131579,7368
1096,Medical-53f6f364,0.668135,0.684211,5972


In [12]:
print("min:", df_eval_loaded["tokens"].min())
print("max:", df_eval_loaded["tokens"].max())
print("avg:", df_eval_loaded["tokens"].mean())
print("sum:", df_eval_loaded["tokens"].sum())

min: 997
max: 12842
avg: 3850.6593806921674
sum: 4228024


In [13]:
print("min:", df_eval_loaded["accuracy"].min())
print("max:", df_eval_loaded["accuracy"].max())
print("avg:", df_eval_loaded["accuracy"].mean())
print("sum:", df_eval_loaded["accuracy"].sum())

min: 0.16753409802913666
max: 0.999999999925
avg: 0.6438947848796382
sum: 706.9964737978428


In [14]:
print("min:", df_eval_loaded["rouge"].min())
print("max:", df_eval_loaded["rouge"].max())
print("avg:", df_eval_loaded["rouge"].mean())
print("sum:", df_eval_loaded["rouge"].sum())

min: 0.044444444444444446
max: 1.0
avg: 0.35954513686712053
sum: 394.78056028009837
